# M1 · Student B — Design/select the compact descriptor architecture
**CSE 681 · Project 26 — Compact Deep Feature Descriptors vs. Classical SIFT/ORB**

| | |
|---|---|
| **Milestone** | M1 — Proposal & Literature Review (Week 3) |
| **Owner** | Student B (Model/Algorithm Lead) |
| **Runtime** | **GPU** (Runtime → Change runtime type → T4 GPU) · about 10 min |

**This notebook produces (saved to the shared Drive folder `DIP_P26/`):**
- `tables/m1_arch_candidates.tex`: size/speed of candidate architectures vs SIFT/ORB (report §4 *Proposed Method*)
- `figures/m1_patch_orientation_check.png`: proof that the patch extraction is rotation-consistent
- `figures/m1_training_pairs.png`: what the training data looks like
- `results/m1_feasibility.json`: measured training speed, used in §5 *Feasibility*

> First time? Read `README.md`: one teammate creates the shared folder `DIP_P26` on Google Drive,
> uploads `p26lib.py` into it and shares it; everyone adds a shortcut to *My Drive*.

In [ ]:
#@title 0 · Setup (run every session)
import os, sys, json, time
SMOKE_TEST = False   # True = 1-2 minute dry run on tiny SYNTHETIC data (to check the code). NEVER report smoke numbers.
SMOKE_TEST = SMOKE_TEST or os.environ.get("P26_SMOKE") == "1"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJ, DATA = "/content/drive/MyDrive/DIP_P26", "/content/data"
except ImportError:                       # running locally / Kaggle
    PROJ = os.environ.get("P26_PROJ", os.path.abspath("DIP_P26"))
    DATA = os.environ.get("P26_DATA", os.path.abspath("data"))
os.makedirs(PROJ, exist_ok=True); os.makedirs(DATA, exist_ok=True)
sys.path.insert(0, PROJ)
assert os.path.exists(os.path.join(PROJ, "p26lib.py")), f"Upload p26lib.py into {PROJ} first (see README)."

import importlib, p26lib as P; importlib.reload(P)
import numpy as np, pandas as pd, cv2, torch
import matplotlib; import matplotlib.pyplot as plt
P.set_seed(P.SEED)
D = P.project_dirs(PROJ)
TAG = "SMOKE_" if SMOKE_TEST else ""          # smoke-test outputs never overwrite real ones
def OUT(kind, name):                           # e.g. OUT("tables", "m1_baseline.tex")
    return os.path.join(D[kind], TAG + name)
DEVICE = P.get_device()
print("SMOKE_TEST =", SMOKE_TEST, "| device =", DEVICE)
print(json.dumps(P.env_info(), indent=1))

## 1 · Design idea (for report §4)
**Pipeline:** SIFT keypoints (location, scale, orientation) → a 32×32 oriented, scale-normalised patch → **compact CNN** → L2-normalised descriptor → MNN matching → RANSAC.

Both methods use the same detector, so any difference against SIFT comes **only from the descriptor**.

**Architecture:** an L2-Net/HardNet-style fully-convolutional network (3×3 conv, BN, ReLU; two stride-2 stages; final 8×8 conv), shrunk by a width factor `w`.
HardNet itself is `w=32, dim=128` (~1.3 M parameters). Our default candidate is `w=16, dim=64`, which is about 4× fewer parameters and FLOPs and a
**256 B** descriptor (SIFT uses 512 B as float32).

**Loss:** HardNet hardest-in-batch triplet margin loss.

**Data:** Brown/UBC PhotoTour patches (the standard HardNet training set), with self-supervised homography-warped pairs from COCO as a fallback. **Nothing from HPatches is used for training.**

**Extension over the baseline (what is new):** an explicit accuracy / speed / memory trade-off study (width × dimension), plus a **binarised** variant that competes with ORB at the same 32 B.

In [ ]:
#@title 2 · Candidate architectures: params, MACs, bytes, CPU/GPU speed -> tables/m1_arch_candidates.tex
P.set_threads(1)                       # timing protocol: 1 CPU thread for everybody (see M1_C)
rows = []
cands = [(8, 64), (16, 32), (16, 64), (16, 128), (32, 128)]
x_cpu = torch.rand(1000, 1, 32, 32)
for w, d in cands:
    m = P.CompactDescriptor(w, d).eval()
    with torch.no_grad():
        m(x_cpu[:64]); t0 = time.perf_counter(); m(x_cpu); cpu_ms = 1e3 * (time.perf_counter() - t0)
    gpu_ms = float("nan")
    if torch.cuda.is_available():
        mg, xg = m.to("cuda"), x_cpu.to("cuda")
        with torch.no_grad():
            mg(xg); torch.cuda.synchronize(); t0 = time.perf_counter(); mg(xg); torch.cuda.synchronize()
        gpu_ms = 1e3 * (time.perf_counter() - t0)
    rows.append({"model": f"CNN w={w} d={d}" + (" (HardNet size)" if (w, d) == (32, 128) else ""),
                 "params (K)": P.count_params(m) / 1e3, "MMACs/patch": P.count_macs(m) / 1e6,
                 "B/desc float": d * 4, "B/desc binary": d // 8,
                 "CPU ms /1000 patches": cpu_ms, "GPU ms /1000 patches": gpu_ms})
rows += [{"model": "SIFT (OpenCV)", "params (K)": 0, "MMACs/patch": float("nan"), "B/desc float": 512, "B/desc binary": float("nan"),
          "CPU ms /1000 patches": float("nan"), "GPU ms /1000 patches": float("nan")},
         {"model": "ORB (OpenCV)", "params (K)": 0, "MMACs/patch": float("nan"), "B/desc float": float("nan"), "B/desc binary": 32,
          "CPU ms /1000 patches": float("nan"), "GPU ms /1000 patches": float("nan")}]
arch = pd.DataFrame(rows)
display(arch.round(2))
P.df_to_latex(arch.round(2), OUT("tables", "m1_arch_candidates.tex"),
              "Candidate compact descriptors (input 32x32 patch). CPU timing: 1 thread, batch of 1000.", "tab:arch", fmt="{:.1f}")
P.set_threads(os.cpu_count())
print(P.CompactDescriptor(16, 64))

## 3 · Patch extraction sanity check
If the patch orientation convention were wrong, the CNN would never become rotation invariant. To check it, we rotate an image by a known angle, detect
SIFT keypoints in both versions, extract patches and correlate the patches of corresponding keypoints. With the correct convention the NCC should be
close to 1.

In [ ]:
#@title 3 · Rotation-consistency check -> figures/m1_patch_orientation_check.png
img = P._sample_images()[0]
h, w = img.shape
M = cv2.getRotationMatrix2D((w / 2, h / 2), 40, 1.0); H = np.vstack([M, [0, 0, 1]])
img2 = cv2.warpAffine(img, M, (w, h))
sift = cv2.SIFT_create(800)
k1, k2 = sift.detect(img, None), sift.detect(img2, None)
proj = P.warp_points(np.array([k.pt for k in k1]), H); c2 = np.array([k.pt for k in k2])
dist = np.linalg.norm(proj[:, None] - c2[None], axis=2); j = dist.argmin(1)
pairs = [(i, j[i]) for i in np.where(dist.min(1) < 1)[0] if abs(k1[i].size - k2[j[i]].size) / k1[i].size < 0.1]
A, B = [k1[i] for i, _ in pairs], [k2[jj] for _, jj in pairs]
pa = P.extract_patches(img, *P.keypoint_frames(A)); pb = P.extract_patches(img2, *P.keypoint_frames(B))
ncc = [np.corrcoef(a.ravel().astype(float), b.ravel().astype(float))[0, 1] for a, b in zip(pa, pb)]
print(f"{len(pairs)} corresponding keypoints, median NCC of their patches = {np.median(ncc):.3f} (should be > 0.9)")
assert np.median(ncc) > 0.8, "patch orientation convention is broken"
fig, axs = plt.subplots(2, 10, figsize=(15, 3.2))
for c in range(10):
    axs[0, c].imshow(pa[c], cmap="gray"); axs[1, c].imshow(pb[c], cmap="gray")
    axs[0, c].axis("off"); axs[1, c].axis("off")
axs[0, 0].set_title("image", loc="left"); axs[1, 0].set_title("rotated 40°", loc="left")
plt.tight_layout(); plt.savefig(OUT("figures", "m1_patch_orientation_check.png"), dpi=150); plt.show()

## 4 · Training data check
The plan is **PhotoTour/liberty** for training and **notredame** for validation (FPR95), which is the HardNet protocol. If the PhotoTour server is down,
the notebook falls back to self-supervised pairs made by warping COCO val2017 images with random homographies (`PairSampler.from_images`).
Either way, **no HPatches images are used for training**.

In [ ]:
#@title 4 · Load a small training set + show anchor/positive pairs -> figures/m1_training_pairs.png
TRAIN_SOURCE = "phototour"     # "phototour" or "homography"
if SMOKE_TEST:
    TRAIN_SOURCE = "homography"
sampler = None
if TRAIN_SOURCE == "phototour":
    try:
        sampler = P.PairSampler.from_phototour(os.path.join(DATA, "phototour"), "liberty", max_patches=100_000)
    except Exception as e:
        print("PhotoTour unavailable -> falling back to homography pairs:", repr(e)[:200])
if sampler is None:
    imgs = P._sample_images() if SMOKE_TEST else P.load_image_folder(P.download_coco_val(DATA), max_images=300)
    sampler = P.PairSampler.from_images(imgs, pairs_per_image=200)
a, p = sampler.sample(12, np.random.default_rng(P.SEED))
fig, axs = plt.subplots(2, 12, figsize=(15, 2.8))
for c in range(12):
    axs[0, c].imshow(a[c], cmap="gray"); axs[1, c].imshow(p[c], cmap="gray"); axs[0, c].axis("off"); axs[1, c].axis("off")
axs[0, 0].set_title("anchor", loc="left"); axs[1, 0].set_title("positive", loc="left")
plt.suptitle(f"Training pairs ({sampler.name})"); plt.tight_layout()
plt.savefig(OUT("figures", "m1_training_pairs.png"), dpi=150); plt.show()

In [ ]:
#@title 5 · Feasibility: measure training speed and extrapolate the M2 budget -> results/m1_feasibility.json
model = P.CompactDescriptor(16, 64)
STEPS, BATCH = (10, 64) if SMOKE_TEST else (100, 512)
hist = P.train_descriptor(model, sampler, epochs=1, steps_per_epoch=STEPS, batch_size=BATCH, device=DEVICE)
sec_per_step = hist[0]["epoch_sec"] / STEPS
plan = {"epochs": 20, "steps_per_epoch": 1000, "batch": 512}
total_min = sec_per_step * plan["epochs"] * plan["steps_per_epoch"] * (512 / BATCH) / 60
feas = {"device": str(DEVICE), "gpu": P.env_info()["gpu"], "sec_per_step_batch512": sec_per_step * 512 / BATCH,
        "planned": plan, "estimated_training_minutes": total_min}
json.dump(feas, open(OUT("results", "m1_feasibility.json"), "w"), indent=1)
print(json.dumps(feas, indent=1))
print(f"\n-> Report sentence: 'Training the w=16/d=64 descriptor for {plan['epochs']}x{plan['steps_per_epoch']} steps "
      f"(batch 512) takes about {total_min:.0f} min on {feas['gpu'] if feas['gpu']!='none' else 'CPU'}; evaluation on all of "
      f"HPatches takes a few minutes. The project is GPU-light and fits a free Colab T4 or a laptop.'")

## ✅ Deliverable checklist & where it goes in the report

- [ ] **§4 Proposed Method**: copy the "Design idea" text above and adapt it. Add `tables/m1_arch_candidates.tex` and draw a small pipeline diagram (draw.io, or TikZ).
- [ ] **§5 Feasibility**: the sentence printed by cell 5, compute tier **GPU-light (CPU-OK)**
- [ ] **MVP** (§5.1): *"A w=16/d=64 float descriptor trained on PhotoTour, compared against SIFT and ORB on all 108 HPatches sequences (MMA, H-acc, time, bytes)."*
- [ ] **Stretch** (§5.2): *"(i) width × dimension speed ablation, (ii) a binary 64-bit/128-bit variant against ORB at equal or lower memory, (iii) a registration demo."*
- [ ] **Risks** (§6): training doesn't beat SIFT → fine-tune pretrained HardNet / report an honest negative result; CNN too slow on CPU → smaller width, report the trade-off curve; PhotoTour download fails → homography-pair fallback (already implemented)